# 20261108 · 实战项目串讲：LangGraph SQL Agent

> 项目目录：`week24/langgraph-sql-agent/`
> 数据：`week24/employees.sql`（Navicat 导出的员工库，六张表，约 305 MB / 392 万行）
> 依赖：`langgraph 1.2.18`、`langchain 1.4.7`、`sqlglot 30.30.0`、`PyMySQL 1.2.3`

**一句话**：用户问一句中文，图里走一遍「识别意图 → 取 schema → 写 SQL → 校验 → EXPLAIN → 执行 → 校验结果」，任何一步不对就带着错误原因回到草稿节点重写，最多三次。

和前三讲的关系：

| 课程 | 这个项目用在哪 |
|---|---|
| 1104 条件边 / 循环 | `repair_or_finish()` 的三条回边，`draft_sql ↔ check_sql ↔ execute_sql` |
| 1104 `TypedDict` 部分更新 | `AgentState(TypedDict, total=False)`，每个节点只返回自己那几个键 |
| 1106 `stream_mode` | `main()` 用 `stream_mode="updates"` 逐节点打印 |
| 1106 `ToolNode` | **没用**——这里「执行」是固定步骤，不是模型选的工具，所以写成普通节点 |
| 1106 `interrupt()` | 也没用（本项目最值得补的一处，见第 10 节待办） |

## 执行说明

| 项 | 说明 |
|---|---|
| 内核 | `langchain-learning (3.11.9)`（`week23/.venv`） |
| 额外依赖 | 本机把 `sqlglot`/`pymysql` 放在 `E:\Codex\2026-11-04\new-chat\work\sql_agent_deps`，第 1 格会把它和项目目录一起加进 `sys.path` |
| 数据库 | Docker 容器 MySQL 8.4，`127.0.0.1:13307`，库名 `47`；第 1 格自动探测，探不到就给「离线降级」提示 |
| 模型 | 第 4 节用 `week23/20261027/lab_config.py` 的 `get_model()`（千问） |

> **证据边界**：凡打印真实查询结果的格子，都在本机 MySQL 上真跑过；凡用 `StubReader` 的格子是**测试替身**（和项目自带 `verify.py` 里 `OneBadResult` 同一手法），不连库、输出固定。


## 1. 先把项目「装」进内核

三件事：把项目目录加进 `sys.path`（为了 `import agent`）、把额外依赖目录加进去、加载数据库连接配置。

`agent.py` 在 `import` 阶段不读环境变量，所以先导入再探测数据库是安全的。


In [1]:
import json
import os
import pathlib
import sys
import time
from collections import Counter
from dataclasses import replace
from importlib.metadata import version

PROJECT = pathlib.Path(r"d:\project\step1\week24\langgraph-sql-agent")
DEPS = pathlib.Path(r"E:\Codex\2026-11-04\new-chat\work\sql_agent_deps")
ENV_CANDIDATES = [
    PROJECT / ".env",
    pathlib.Path(r"E:\Codex\2026-11-04\new-chat\work\sql_agent_local.env"),
]

for p in (PROJECT, DEPS):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dotenv import load_dotenv

ENV_FILE = next((p for p in ENV_CANDIDATES if p.is_file()), None)
if ENV_FILE:
    load_dotenv(ENV_FILE, override=True)

print("langgraph :", version("langgraph"))
print("langchain :", version("langchain"))
print("sqlglot   :", version("sqlglot"))
print("pymysql   :", version("pymysql"))
print("项目目录  :", PROJECT, "| 存在:", PROJECT.is_dir())
print("依赖目录  :", DEPS, "| 存在:", DEPS.is_dir())
print("连接配置  :", ENV_FILE.name if ENV_FILE else "未找到（数据库相关格子会跳过）")


langgraph : 1.2.18
langchain : 1.4.7
sqlglot   : 30.30.0
pymysql   : 1.2.3
项目目录  : d:\project\step1\week24\langgraph-sql-agent | 存在: True
依赖目录  : E:\Codex\2026-11-04\new-chat\work\sql_agent_deps | 存在: True
连接配置  : sql_agent_local.env


In [2]:
import agent as A
from db import TABLES, connect, database_name

print("允许访问的表：", TABLES)
print("返回行数上限：", A.MAX_ROWS)
print("函数白名单  ：", len(A.SAFE_FUNCTIONS), "项")
print("验收题      ：", A.QUESTION)
print("验收 SQL    ：", A.COUNT_SQL)

# 探测数据库是否可用
DB_READY = False
DB_ERROR = ""
if ENV_FILE:
    try:
        reader_ping = A.MySQLReader()
        _schema = reader_ping.schema(["employees"])
        DB_READY = bool(_schema)
    except Exception as exc:                      # noqa: BLE001
        DB_ERROR = f"{type(exc).__name__}: {exc}"[:160]

print("\n数据库可用：", DB_READY, "| 库名:", database_name() if ENV_FILE else "-")
if DB_ERROR:
    print("探测失败：", DB_ERROR)


允许访问的表： ('employees', 'departments', 'dept_emp', 'dept_manager', 'salaries', 'titles')
返回行数上限： 50
函数白名单  ： 24 项
验收题      ： 1987 年之后入职的员工人数是多少？
验收 SQL    ： SELECT COUNT(*) AS employee_count FROM employees WHERE hire_date >= '1988-01-01'

数据库可用： True | 库名: 47


## 2. 架构：七个节点、五条条件边

```mermaid
flowchart LR
    S((START)) --> I[recognize_intent]
    I -->|rejected| ANS[answer]
    I -->|ok| SC[search_schema]
    SC -->|failed| ANS
    SC -->|ok| DS[draft_sql]
    DS --> CS[check_sql]
    CS -->|error 未耗尽| DS
    CS -->|error 耗尽| ANS
    CS -->|ok| ES[execute_sql]
    ES -->|error 未耗尽| DS
    ES -->|error 耗尽| ANS
    ES -->|ok| CR[check_result]
    CR -->|异常未耗尽| DS
    CR -->|异常耗尽| ANS
    CR -->|ok| ANS
    ANS --> E((END))
```

| 节点 | 做什么 | 关键返回键 |
|---|---|---|
| `recognize_intent` | 抽 1～1000 字校验 → 模型判「是否只读可回答」+ 形态 + 相关表 | `intent` / `employee_1987` / `status` |
| `search_schema` | 从 `information_schema.COLUMNS` 取表结构 | `schema` |
| `draft_sql` | 生成或**修订**草稿；`attempt += 1`，同时清空上一轮 `sql`/`error` | `attempt` / `draft` |
| `check_sql` | `validate_sql()`：sqlglot 语法树白名单 + `qualify()` 查未知列 | `sql`（**重写后的**）或 `error` |
| `execute_sql` | `EXPLAIN` + 只读事务执行，取前 51 行 | `result`（`QueryResult`） |
| `check_result` | `result_error()`：列行结构、count 语义 | `status=success` 或 `error` |
| `answer` | 本地模板组织答复（不让模型编数字） | `answer` |

> 注意 `check_sql` 返回的是 `qualify()` **重写后**的 SQL —— 执行的不是模型原文。这一点在第 6 节会打印对比。


In [3]:
# 图的节点与边（直接读项目里编译好的图，不用手抄）
EMP_SCHEMA = A.MySQLReader().schema(["employees"]) if DB_READY else None

if not DB_READY:
    # 离线时给一份与真实库一致的字段名，供校验器使用（类型按 employees 表定义）
    EMP_SCHEMA = {"employees": {
        "emp_no": {"type": "int", "key": "PRI", "comment": ""},
        "birth_date": {"type": "date", "key": "", "comment": ""},
        "first_name": {"type": "varchar", "key": "", "comment": ""},
        "last_name": {"type": "varchar", "key": "", "comment": ""},
        "gender": {"type": "enum", "key": "", "comment": ""},
        "hire_date": {"type": "date", "key": "", "comment": ""},
    }}
    print("⚠️ 未连上数据库，使用内置 employees 字段表（仅供参考）")

demo_graph = A.build_graph(A.MySQLReader(), A.ScriptedPlanner()) if DB_READY else None
if demo_graph:
    g = demo_graph.get_graph()
    print("节点：", [n for n in g.nodes if not n.startswith("__")])
    print("\n边：")
    for e in g.edges:
        kind = "条件边" if getattr(e, "conditional", False) else "普通边"
        print(f"  {e.source:>16s} → {e.target:<16s} [{kind}]")


节点： ['recognize_intent', 'search_schema', 'draft_sql', 'check_sql', 'execute_sql', 'check_result', 'answer']

边：
         __start__ → recognize_intent [普通边]
      check_result → answer           [条件边]
      check_result → draft_sql        [条件边]
         check_sql → answer           [条件边]
         check_sql → draft_sql        [条件边]
         check_sql → execute_sql      [条件边]
         draft_sql → check_sql        [普通边]
       execute_sql → answer           [条件边]
       execute_sql → check_result     [条件边]
       execute_sql → draft_sql        [条件边]
  recognize_intent → answer           [条件边]
  recognize_intent → search_schema    [条件边]
     search_schema → answer           [条件边]
     search_schema → draft_sql        [条件边]
            answer → __end__          [普通边]


## 3. 状态设计：`total=False` + 只返回自己那几个键

```python
class AgentState(TypedDict, total=False):
    question: str
    intent: dict
    schema: dict
    employee_1987: bool
    attempt: int
    draft: str
    sql: str
    error: str
    status: Literal["running", "success", "rejected", "failed"]
    result: QueryResult
    answer: str
    failure_stage: str
```

这正是 1104 第 1 节讲的「节点返回部分状态」在真实项目里的用法：

| 设计选择 | 为什么 |
|---|---|
| `total=False` | 所有键都可缺省：拒绝的分支根本不会有 `schema`、`result` |
| 每个节点只返回自己改的键 | 不会误覆盖上游数据；也天然形成「谁改了什么」的日志 |
| `draft` 与 `sql` 分开 | `draft` 是模型原文，`sql` 是校验并重写后真正执行的语句 |
| `attempt` 单独计数 | 重试上限与「生成次数」解耦，避免靠 `len(messages)` 之类猜 |

下一格用 `stream_mode="updates"` 看每个节点到底返回了哪些键——这也是排查「某个字段为什么没更新」的最快办法。


In [4]:
def run_path(reader, planner, question=A.QUESTION, max_attempts=3):
    """跑一遍图，收集节点路径 + 每步返回键 + 最终状态 + 失败阶段。"""
    graph = A.build_graph(reader, planner, max_attempts)
    path, keys, stages, final = [], [], [], {}
    for chunk in graph.stream({"question": question}, stream_mode="updates",
                              config={"recursion_limit": 32}):
        for node, update in chunk.items():
            path.append(node)
            keys.append((node, list(update)))
            if update.get("failure_stage"):
                stages.append(update["failure_stage"])
            final.update(update)
    return path, keys, final, stages


if DB_READY:
    path, keys, final, stages = run_path(A.MySQLReader(), A.ScriptedPlanner())
    for node, ks in keys:
        print(f"{node:16s} 返回键 = {ks}")
    print("\n路径：", " → ".join(path))
    print("answer：", final["answer"])
else:
    print("跳过：需要数据库。")


recognize_intent 返回键 = ['intent', 'employee_1987', 'status']
search_schema    返回键 = ['schema']
draft_sql        返回键 = ['attempt', 'draft', 'sql', 'error', 'failure_stage']
check_sql        返回键 = ['sql']
execute_sql      返回键 = ['result']
check_result     返回键 = ['status']
answer           返回键 = ['answer']

路径： recognize_intent → search_schema → draft_sql → check_sql → execute_sql → check_result → answer
answer： 1987 年之后（含 1988-01-01 当天）入职的员工共有 195,057 人。


## 4. 意图识别：结构化输出 + 一条「口径硬编码」

`ModelPlanner` 用 `with_structured_output` 把模型输出约束成 pydantic：

```python
class Intent(BaseModel):
    allowed: bool
    shape: Literal["count", "table"]
    tables: list[str]
```

两个细节值得注意：

| 细节 | 代码 | 说明 |
|---|---|---|
| 形态只有两种 | `count` = 单个总数；`table` = 其余（分组、列表） | 形态决定后面两处校验的严格程度 |
| **口径硬编码** | `recognize_intent` 里 `if is_verification_question(question): intent = Intent(allowed=True, shape="count", tables=["employees"])` | 命中正则就无视模型判断，强制成验收题的形态 |

第二点是有意的「作弊」：保证验收题结果稳定。代价是——**换个说法就绕过去了**，第 4.2 格会实测。


In [5]:
t0 = time.perf_counter()
course_model = A.load_course_model(pathlib.Path(r"d:\project\step1\week23\20261027"))
planner_real = A.ModelPlanner(course_model)
print(f"模型就绪，耗时 {time.perf_counter() - t0:.1f}s\n")

cases = [
    "1987 年之后入职的员工人数是多少？",
    "按性别统计员工人数",
    "把 employees 表删掉",
    "今天成都天气怎么样",
]
for q in cases:
    t0 = time.perf_counter()
    intent = planner_real.classify(q)
    hit = A.is_verification_question(q)
    print(f"问题：{q}")
    print(f"  Intent(allowed={intent.allowed}, shape={intent.shape!r}, tables={intent.tables})"
          f"  正则命中={hit}  {time.perf_counter() - t0:.1f}s")


模型就绪，耗时 1.3s

问题：1987 年之后入职的员工人数是多少？
  Intent(allowed=True, shape='count', tables=['employees'])  正则命中=True  1.1s
问题：按性别统计员工人数
  Intent(allowed=True, shape='table', tables=['employees'])  正则命中=False  0.8s
问题：把 employees 表删掉
  Intent(allowed=False, shape='count', tables=['employees'])  正则命中=False  0.9s
问题：今天成都天气怎么样
  Intent(allowed=False, shape='table', tables=[])  正则命中=False  1.2s


In [6]:
# 口径作弊的实测：同一件事，换三种说法，正则命中情况完全不同
variants = [
    "1987 年之后入职的员工人数是多少？",
    "1988 年以后入职的员工有多少人？",
    "帮我数一下 1988 年 1 月 1 日之后入职的人",
]
for q in variants:
    print(f"{A.is_verification_question(q)!s:5s} ← {q}")

print("\n未命中的问题走「通用路径」：不做 hire_date >= '1988-01-01' 的边界校验，")
print("只做通用的单表 COUNT 结构检查。也就是说验收口径可以被措辞绕过。")


True  ← 1987 年之后入职的员工人数是多少？
False ← 1988 年以后入职的员工有多少人？
False ← 帮我数一下 1988 年 1 月 1 日之后入职的人

未命中的问题走「通用路径」：不做 hire_date >= '1988-01-01' 的边界校验，
只做通用的单表 COUNT 结构检查。也就是说验收口径可以被措辞绕过。


## 5. 检索 schema：只给模型看「这几张表的这几个字段」

`MySQLReader.schema()` 直接查 `information_schema.COLUMNS`，只保留 `search_schema` 选中的表：

```python
candidates = list(dict.fromkeys(t for t in intent["tables"] if t in TABLES))[:4]
if state["employee_1987"]:
    candidates = ["employees"]          # 验收题强制单表
if not candidates:
    candidates = ["employees"]          # ← 兜底
```

| 设计 | 好处 | 风险 |
|---|---|---|
| 只给相关表 | prompt 小、模型不易瞎 JOIN | 模型漏选表 → 后面的校验必然拒（安全，但要多跑一轮） |
| 最多 4 张表 | 控制上下文长度 | 六表全要的问题会缺表 |
| 兜底 `["employees"]` | 保证不会空手进草稿节点 | **会掩盖意图误分类**：`allowed=True` 但 `tables=[]` 时，无关问题也会拿员工表硬查 |


In [7]:
if DB_READY:
    reader = A.MySQLReader()
    schema = reader.schema(["employees"])
    for table, columns in schema.items():
        print(f"[{table}] {len(columns)} 个字段")
        for name, props in columns.items():
            key = f" {props['key']}" if props["key"] else ""
            print(f"  {name:12s} {props['type']:10s}{key}")
else:
    print("跳过：需要数据库。")


[employees] 6 个字段
  emp_no       int        PRI
  birth_date   date      
  first_name   varchar   
  last_name    varchar   
  gender       enum      
  hire_date    date      


## 6. SQL 校验：整个项目最核心的一段

`validate_sql()` 做六件事，顺序很重要（先便宜的、后贵的）：

| 顺序 | 检查 | 实现 | 拦住什么 |
|---|---|---|---|
| 1 | 长度与注释 | `len > 4000`、正则查 `--` `/*` `#` `\x00` | 注释注入、超长 payload |
| 2 | 单条 SELECT | `parse(draft, read="mysql")`，`len == 1 and isinstance(exp.Select)` | 多语句、`DELETE`/`UPDATE`、`SHOW` |
| 3 | 危险节点 | `walk()` 找 `CTE / Subquery / SetOperation / Into / Lock / Parameter / SessionParameter / PropertyEQ / Dot / Window` | `INTO OUTFILE`、`FOR UPDATE`、`@var :=`、窗口函数 |
| 4 | 表 | `find_all(exp.Table)`，必须都在 schema 里且不带库名 | 跨库 `mysql.user`、未检索到的表 |
| 5 | 函数 | `find_all(exp.Func)`，`sql_name()` 必须在 `SAFE_FUNCTIONS` | `SLEEP()`、`LOAD_FILE()` 这类 |
| 6 | 形态 + 列 | `count` 形态必须是单个 `COUNT`；`_verify_employee_count()` 再查日期边界 | 验收题写成别的聚合 |

最后还有一步 `sqlglot.optimizer.qualify()`：**在执行之前**就把不存在的列、无法消歧的列找出来——比让 MySQL 报 1054 更早、也更省事。

### 读校验器输出时要注意的两处

| 现象 | 原因 | 建议 |
|---|---|---|
| `SLEEP(1)` 报成 `不允许使用函数 ANONYMOUS` | sqlglot 把不认识的函数归为 `exp.Anonymous`，它的 `sql_name()` 就是 `"ANONYMOUS"`，不是 `SLEEP` | 报错时用 `getattr(node, "name", node.sql_name())` 取真实函数名 |
| `INTO OUTFILE` 报 `ParseError` 而不是被 `exp.Into` 拦下 | MySQL 方言解析不了这个语法，在 **parse 阶段**就失败了 | 同样是拦截，但要知道它发生在第 2 步而不是第 3 步 |
| `UNION` 报 `只允许一条 SELECT` 而不是被 `exp.SetOperation` 拦下 | `sqlglot` 把它解析成 `exp.Union`（不是 `exp.Select`），第 2 步就出局 | 黑名单是第二道防线；实际生效的往往是第 2 步 |

> 这两条不影响安全性，只影响「失败原因是否好读」——而失败原因会原样喂回 `draft_sql` 给模型看，所以可读性直接决定重试能不能改对。

下面两格：先跑对抗样例（分两组，因为 `count` 形态会先跑验收口径），再对比「草稿 SQL」与「实际执行 SQL」。


In [8]:
COUNT_CASES = {
    "多语句": A.COUNT_SQL + "; SELECT * FROM employees",
    "写操作": "DELETE FROM employees",
    "跨库": "SELECT COUNT(*) FROM mysql.user",
    "导出文件": "SELECT emp_no FROM employees INTO OUTFILE 'blocked.txt'",
    "错误日期": A.COUNT_SQL.replace("1988-01-01", "1987-01-01"),
    "注释截断": A.COUNT_SQL + " -- 后面都忽略",
}

TABLE_CASES = {
    "未知列": A.COUNT_SQL.replace("hire_date", "unknown_column"),
    "休眠函数": "SELECT SLEEP(1) FROM employees",
    "DATEDIFF": "SELECT emp_no FROM employees WHERE DATEDIFF('2000-01-01', hire_date) > 3650",
    "IN 子查询": "SELECT emp_no FROM employees WHERE emp_no IN (SELECT emp_no FROM titles)",
    "窗口函数": "SELECT ROW_NUMBER() OVER (ORDER BY emp_no) FROM employees",
    "UNION": "SELECT emp_no FROM employees UNION SELECT emp_no FROM titles",
}


def check(cases, shape, employee_1987, title):
    print(f"--- {title}（shape={shape!r}, 验收口径={employee_1987}）---")
    blocked = 0
    for name, sql in cases.items():
        try:
            out = A.validate_sql(sql, EMP_SCHEMA, shape, employee_1987=employee_1987)
            print(f"  ❌ 通过  {name:9s} → {out[:60]}")
        except Exception as exc:                  # noqa: BLE001
            blocked += 1
            msg = str(exc).replace("\n", " ")[:62]
            print(f"  ✅ 拦截  {name:9s} | {type(exc).__name__}: {msg}")
    print(f"  小计：拦截 {blocked} / {len(cases)}\n")
    return blocked


total = check(COUNT_CASES, "count", True, "验收口径组")
total += check(TABLE_CASES, "table", False, "通用路径组")
print(f"合计拦截 {total} / {len(COUNT_CASES) + len(TABLE_CASES)} 条")


--- 验收口径组（shape='count', 验收口径=True）---
  ✅ 拦截  多语句       | ValueError: 只允许一条 SELECT
  ✅ 拦截  写操作       | ValueError: 只允许一条 SELECT
  ✅ 拦截  跨库        | ValueError: SQL 引用了未检索到的表或数据库
  ✅ 拦截  导出文件      | ParseError: Invalid expression / Unexpected token. Line 1, Col: 33.   SELE
  ✅ 拦截  错误日期      | ValueError: 验收题请使用 hire_date >= '1988-01-01'，含当天
  ✅ 拦截  注释截断      | ValueError: SQL 超长或包含注释
  小计：拦截 6 / 6

--- 通用路径组（shape='table', 验收口径=False）---
  ✅ 拦截  未知列       | OptimizeError: Column 'unknown_column' could not be resolved. Line: 1, Col: 6
  ✅ 拦截  休眠函数      | ValueError: 不允许使用函数 ANONYMOUS
  ✅ 拦截  DATEDIFF  | ValueError: 不允许使用函数 DATEDIFF
  ✅ 拦截  IN 子查询    | ValueError: 不支持子查询、CTE、集合操作、变量、窗口函数或带副作用的子句
  ✅ 拦截  窗口函数      | ValueError: 不支持子查询、CTE、集合操作、变量、窗口函数或带副作用的子句
  ✅ 拦截  UNION     | ValueError: 只允许一条 SELECT
  小计：拦截 6 / 6

合计拦截 12 / 12 条


In [9]:
print("草稿（模型写的）：")
print(" ", A.COUNT_SQL)

for label, shape, emp in [("count 形态", "count", True), ("table 形态", "table", False)]:
    print(f"\n执行（validate_sql 重写后的 {label}）：")
    print(" ", A.validate_sql(A.COUNT_SQL, EMP_SCHEMA, shape, employee_1987=emp))

print("\nLIMIT 规则（table 形态）：")
no_limit = "SELECT emp_no, hire_date FROM employees ORDER BY hire_date"
print("  无 LIMIT  →", A.validate_sql(no_limit, EMP_SCHEMA, "table")[-40:])
with_limit = no_limit + " LIMIT 5"
print("  已有 5    →", A.validate_sql(with_limit, EMP_SCHEMA, "table")[-30:])
over_limit = no_limit + " LIMIT 500"
print("  写了 500 →", A.validate_sql(over_limit, EMP_SCHEMA, "table")[-40:])


草稿（模型写的）：
  SELECT COUNT(*) AS employee_count FROM employees WHERE hire_date >= '1988-01-01'

执行（validate_sql 重写后的 count 形态）：
  SELECT COUNT(*) AS `employee_count` FROM `employees` AS `employees` WHERE `employees`.`hire_date` >= '1988-01-01'

执行（validate_sql 重写后的 table 形态）：
  SELECT COUNT(*) AS `employee_count` FROM `employees` AS `employees` WHERE `employees`.`hire_date` >= '1988-01-01' LIMIT 51

LIMIT 规则（table 形态）：
  无 LIMIT  → employees` ORDER BY `hire_date` LIMIT 51
  已有 5    → ` ORDER BY `hire_date` LIMIT 5
  写了 500 → employees` ORDER BY `hire_date` LIMIT 51


## 7. EXPLAIN 与只读执行

```python
cursor.execute("SET SESSION MAX_EXECUTION_TIME = 5000")
cursor.execute("START TRANSACTION READ ONLY")
cursor.execute("EXPLAIN " + sql)          # 先看计划，不碰数据
...
cursor.execute(sql)
rows = list(cursor.fetchmany(MAX_ROWS + 1))   # 多取 1 行用来判断是否被截断
connection.rollback()                     # 只读事务也统一回滚
```

| 手段 | 挡住什么 |
|---|---|
| 独立只读账号（`connect()` 拒绝 root） | 即使 SQL 校验被绕过，数据库层还会拒绝写 |
| `START TRANSACTION READ ONLY` | 事务级只读，双保险 |
| `MAX_EXECUTION_TIME = 5000` | 慢查询 5 秒被 MySQL 掐断 |
| `fetchmany(51)` | 结果集不会把内存撑爆 |
| `connect_timeout/read_timeout/write_timeout` | 网络卡住不会永远挂着 |


In [10]:
if DB_READY:
    reader = A.MySQLReader()
    exec_sql = A.validate_sql(A.COUNT_SQL, EMP_SCHEMA, "count", employee_1987=True)
    result = reader.explain_and_execute(exec_sql)
    print("EXPLAIN：")
    for row in result.plan:
        print("  ", json.dumps(row, default=str, ensure_ascii=False))
    print("\ncolumns =", result.columns)
    print("rows    =", result.rows, "| truncated =", result.truncated)

    print("\n只读账号尝试写入（WHERE 1=0，即使有权限也不会改数据）：")
    try:
        with connect() as conn, conn.cursor() as cur:
            cur.execute("UPDATE employees SET hire_date = hire_date WHERE 1 = 0")
    except Exception as exc:                      # noqa: BLE001
        print("  被拒绝 →", type(exc).__name__, getattr(exc, "args", ""))
    else:
        print("  ⚠️ 居然成功了，说明查询账号权限过大")
else:
    print("跳过：需要数据库。")


EXPLAIN：
   {"id": 1, "select_type": "SIMPLE", "table": "employees", "partitions": null, "type": "ALL", "possible_keys": null, "key": null, "key_len": null, "ref": null, "rows": 299556, "filtered": 33.33, "Extra": "Using where"}

columns = ['employee_count']
rows    = [(195057,)] | truncated = False

只读账号尝试写入（WHERE 1=0，即使有权限也不会改数据）：
  被拒绝 → OperationalError (1142, "UPDATE command denied to user 'sql_agent_reader'@'172.18.0.1' for table 'employees'")


## 8. 结果校验：区分「查不到」和「查错了」

`result_error()` 只看两件事：

1. 结构：有没有列、每行列数是否一致；
2. `count` 语义：必须**单行单列、非负整数**，而且 `COUNT(*) = 0` 是**合法结果**（不是失败）。

第 2 条容易写错。很多实现会把「0 行」当异常，于是「某部门一个人都没有」这种正常问题会被误判为失败并触发无意义重试。

### 实测出来的一个可读性问题

| 输入 | 返回的 `error` |
|---|---|
| 两行结果 | 总数查询必须返回单行单列非负整数，0 也是有效结果 |
| 被截断 | 同上 |
| 类型是字符串 | 同上 |

三类完全不同的错误共用同一句话。这条 `error` 会原样拼进下一轮 prompt，模型只能看到一个笼统提示，很难定位到到底是「行数不对」还是「类型不对」。建议拆成三句，或在提示里带上实际拿到的 `len(rows)` / `type(...)`。


In [11]:
from agent import QueryResult

samples = [
    ("正常单行",     QueryResult(["employee_count"], [(195057,)], [], False)),
    ("零也合法",     QueryResult(["employee_count"], [(0,)], [], False)),
    ("两行",         QueryResult(["n"], [(1,), (2,)], [], False)),
    ("列行数不一致", QueryResult(["a", "b"], [(1,)], [], False)),
    ("被截断",       QueryResult(["n"], [(1,)], [], True)),
    ("类型不是 int", QueryResult(["n"], [("195057",)], [], False)),
    ("没有列",       QueryResult([], [], [], False)),
]

for name, res in samples:
    err = A.result_error(res, "count")
    verdict = "通过" if not err else f"拒绝：{err}"
    print(f"{name:12s} → {verdict}")


正常单行         → 通过
零也合法         → 通过
两行           → 拒绝：总数查询必须返回单行单列非负整数，0 也是有效结果
列行数不一致       → 拒绝：返回的列与行结构不一致
被截断          → 拒绝：总数查询必须返回单行单列非负整数，0 也是有效结果
类型不是 int     → 拒绝：总数查询必须返回单行单列非负整数，0 也是有效结果
没有列          → 拒绝：返回的列与行结构不一致


## 9. 重试回路：一条 `repair_or_finish()` 全搞定

```python
def repair_or_finish(state, next_node):
    if not state.get("error"):
        return next_node
    return "draft_sql" if state["attempt"] < max_attempts else "answer"
```

三个节点（`check_sql` / `execute_sql` / `check_result`）共用同一条策略：

| state 情况 | 路由 |
|---|---|
| 没有 `error` | 去各自的下一站（`execute_sql` / `check_result` / `answer`） |
| 有 `error` 且 `attempt < max_attempts` | 回 `draft_sql`（修订） |
| 有 `error` 且 `attempt >= max_attempts` | 去 `answer`（收尾说失败） |

配合 `draft_sql` 里 `attempt = state.get("attempt", 0) + 1`，最多生成 `max_attempts` 次（默认 3）。

下面用三个**测试替身**跑三条路（与 `verify.py` 同手法，不碰真库）：

| 替身 | 作用 |
|---|---|
| `StubReader` | 假 `MySQLReader`：可配置返回行、可只在第一次返回空结果 |
| `ScriptedPlanner(bad_first_draft=True)` | 第一次给错列名（`join_date`），第二次给对的 |
| `AlwaysBadDraft` | 永远给错列名，用来验证「耗尽后停止」 |


In [12]:
class StubReader:
    """测试替身：接口与 MySQLReader 一致，不连库。"""

    def __init__(self, rows=((195057,),), columns=("employee_count",), empty_first=False):
        self.rows = rows
        self.columns = columns
        self.empty_first = empty_first
        self.calls = 0

    def schema(self, tables):
        return {t: EMP_SCHEMA["employees"] for t in tables if t == "employees"}

    def explain_and_execute(self, sql):
        self.calls += 1
        rows = () if (self.empty_first and self.calls == 1) else self.rows
        return QueryResult(list(self.columns), list(rows), [], False)


class AlwaysBadDraft(A.ScriptedPlanner):
    def generate(self, state):
        return "SELECT COUNT(*) FROM employees WHERE nonexistent >= '1988-01-01'"


def show(reader, planner, label, max_attempts=3):
    path, keys, final, stages = run_path(reader, planner, max_attempts=max_attempts)
    counts = Counter(path)
    print(f"\n=== {label} ===")
    print("  路径：", " → ".join(path))
    print("  重复节点：", {k: v for k, v in counts.items() if v > 1} or "无")
    print("  失败阶段：", stages or "无")
    print("  结果：", final.get("status"), "|", final.get("answer"))


if DB_READY:
    stub_ok = StubReader()
    show(stub_ok, A.ScriptedPlanner(), "① 一次成功")

    stub_bad = StubReader(empty_first=True)
    show(stub_bad, A.ScriptedPlanner(bad_first_draft=True),
         "② 错列名 + 空结果：两次回路")

    show(StubReader(), AlwaysBadDraft(), "③ 永远错：2 次后耗尽", max_attempts=2)

    rejected = A.build_graph(StubReader(), A.ScriptedPlanner()).invoke({"question": "删除员工表"})
    print("\n=== ④ 非只读问题 ===")
    print("  status =", rejected["status"], "| answer =", rejected["answer"])
else:
    print("跳过：需要数据库。")



=== ① 一次成功 ===
  路径： recognize_intent → search_schema → draft_sql → check_sql → execute_sql → check_result → answer
  重复节点： 无
  失败阶段： 无
  结果： success | 1987 年之后（含 1988-01-01 当天）入职的员工共有 195,057 人。

=== ② 错列名 + 空结果：两次回路 ===
  路径： recognize_intent → search_schema → draft_sql → check_sql → draft_sql → check_sql → execute_sql → check_result → draft_sql → check_sql → execute_sql → check_result → answer
  重复节点： {'draft_sql': 3, 'check_sql': 3, 'execute_sql': 2, 'check_result': 2}
  失败阶段： ['SQL 校验', '结果校验']
  结果： success | 1987 年之后（含 1988-01-01 当天）入职的员工共有 195,057 人。

=== ③ 永远错：2 次后耗尽 ===
  路径： recognize_intent → search_schema → draft_sql → check_sql → draft_sql → check_sql → answer
  重复节点： {'draft_sql': 2, 'check_sql': 2}
  失败阶段： ['SQL 校验', 'SQL 校验']
  结果： failed | 尝试 2 次后未能完成查询。验收题请使用 hire_date >= '1988-01-01'，含当天

=== ④ 非只读问题 ===
  status = rejected | answer = 无法处理该问题，请提出员工库内的只读查询


## 10. 收口

### 六层防护（这个项目最值得抄的部分）

| 层 | 手段 | 被绕过的后果 |
|---|---|---|
| 连接 | 独立只读账号，`connect()` 里显式拒绝 root | 无（数据库层兜底） |
| 事务 | `START TRANSACTION READ ONLY` | 无 |
| 资源 | `MAX_EXECUTION_TIME=5000`、`fetchmany(51)`、三个超时 | 慢查询拖垮服务 |
| 语法 | sqlglot 解析 + 单条 SELECT + 危险节点黑名单 | 多语句、写操作、导出文件 |
| 语义 | 表/列白名单 + 函数白名单 + `qualify()` 查未知列 | 跨库、幻觉字段、危险函数 |
| 结果 | 列行一致性 + count 单行单列非负整数 | 把「0」当失败、把字符串当计数 |

### 实测出来的边界与可改进点

| # | 现象 | 证据 | 建议 |
|---|---|---|---|
| 1 | 常用函数缺失 | `DATEDIFF(...)` → `不允许使用函数 DATEDIFF` | 白名单补 `DATEDIFF` / `TIMESTAMPDIFF` / `DATE_ADD` / `EXTRACT`，否则「工龄」类问题必挂 |
| 2 | 白名单有 2 条死条目 | sqlglot 里 `exp.Between`、`exp.Not` **不是** `exp.Func` 子类（`And`/`Or`/`Case` 是） | 删掉即可，别误以为它们受控 |
| 3 | 子查询全禁 | `IN (SELECT ...)` 被 `exp.Subquery` 拦 | 若要支持，只放行 `IN (SELECT ...)` 一种形态 |
| 4 | 验收口径可被措辞绕过 | 「1988 年以后入职有多少人」不命中正则 → 走通用路径，不再校验日期边界 | 口径判断改为「结构校验优先」，或对 `shape=count` 一律要求列名与日期边界 |
| 5 | 兜底掩盖误分类 | `search_schema` 里 `tables` 为空时兜底 `["employees"]` | `allowed=True` 但 `tables=[]` 应直接 `failed` |
| 6 | 执行的不是模型原文 | `qualify()` 输出 `FROM \`employees\` AS \`employees\`` | 日志里区分「草稿」与「执行」，便于排障 |
| 7 | `table` 形态只说行数 | `查询返回 2 条记录。` | 有意的「不交给模型编结论」，可加首行/极值模板补信息量 |
| 8 | 没用 1106 的能力 | 无 checkpointer / `interrupt()` / Store | 全表扫描（`type=ALL` 且 `rows` 大）可加人工确认；schema 可用 Store 缓存 |

### 待办

1. 补函数白名单 + 放行 `IN (SELECT ...)`，让「工龄/排名」类问题可用。
2. 把口径判定从正则改成语义 + 结构双校验，堵掉措辞绕过。
3. 用 1106 的 `interrupt()` 给「高代价查询」加人工确认：`result.plan[0]["rows"]` 超过阈值就停下等人。
4. 用 `Store` 缓存 `information_schema` 结果，避免每次提问都查一遍元数据。
5. 给 `answer()` 加 `table` 形态的摘要模板（行数 + 首行 + 极值），信息量更足。


## 自测题

1. `AgentState` 为什么用 `total=False`？它和「节点只返回部分键」是什么关系？
2. `draft` 和 `sql` 为什么要分成两个字段？
3. `validate_sql()` 的六步顺序里，为什么先查注释和单条 SELECT，最后才做 `qualify()`？
4. `COUNT(*) = 0` 为什么必须判为「合法结果」？判成失败会引发什么行为？
5. `repair_or_finish()` 三处复用，各自的「下一站」分别是什么？
6. `attempt` 在哪里递增？为什么不在 `check_sql` 里递增？
7. 为什么「只读账号」和「SQL 校验」两个都要有，不能只留一个？
8. `is_verification_question()` 命中后会强制改写 `intent`，这个设计解决了什么问题、又带来什么副作用？
9. `fetchmany(MAX_ROWS + 1)` 为什么多取一行？
10. 如果给这个项目加人工审批，应该插在哪个节点之后？为什么不是 `check_sql` 之前？

## 复跑命令

```powershell
# 剧本模型（离线可跑，需数据库）
python agent.py --env-file <你的.env>

# 真模型
python agent.py --lab-dir D:\project\step1\week23\20261027 --env-file <你的.env>

# 验收脚本（权限 / 对抗样例 / 两条回路 / 耗尽）
python verify.py --env-file <你的.env>
```
